# Sales Data Exploration  
**Dataset:** [Sales Dataset (Kaggle)](https://www.kaggle.com/datasets/kyanyoga/sample-sales-data)   
*Exploratory Data Analysis*

In [ ]:
## re
import os
from dotenv import load_dotenv

load_dotenv()
%load_ext sql
%sql $DATABASE_URL

%config SqlMagic.feedback = False
%config SqlMagic.displaylimit = None

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


displaylimit: Value None will be treated as 0 (no limit)

In [29]:
#!python -m pip install python-dotenv

In [ ]:
%%sql
SELECT *
FROM sales_data
LIMIT 10;

order_number,quantity_ordered,price_each,order_line_number,sales,order_date,status,qtr_id,month_id,year_id,product_line,msrp,product_code,customer_name,phone,address_line1,address_line2,city,state,postal_code,country,territory,contact_last_name,contact_first_name,deal_size
10121,34,81.35,5,2765.90,2003-05-07 00:00:00,Shipped,2,5,2003,Motorcycles,95,S10_1678,Reims Collectables,26.47.1555,59 rue de l'Abbaye,None,Reims,None,51100,France,EMEA,Henriot,Paul,Small
10134,41,94.74,2,3884.34,2003-07-01 00:00:00,Shipped,3,7,2003,Motorcycles,95,S10_1678,Lyon Souveniers,+33 1 46 62 7555,27 rue du Colonel Pierre Avia,None,Paris,None,75508,France,EMEA,Da Cunha,Daniel,Medium
10180,29,86.13,9,2497.77,2003-11-11 00:00:00,Shipped,4,11,2003,Motorcycles,95,S10_1678,Daedalus Designs Imports,20.16.1555,"184, chausse de Tournai",None,Lille,None,59000,France,EMEA,Rance,Martine,Small
10188,48,100.00,1,5512.32,2003-11-18 00:00:00,Shipped,4,11,2003,Motorcycles,95,S10_1678,Herkku Gifts,+47 2267 3215,"Drammen 121, PR 744 Sentrum",None,Bergen,None,N 5804,Norway,EMEA,Oeztan,Veysel,Medium
10211,41,100.00,14,4708.44,2004-01-15 00:00:00,Shipped,1,1,2004,Motorcycles,95,S10_1678,Auto Canal Petit,(1) 47.55.6555,"25, rue Lauriston",None,Paris,None,75016,France,EMEA,Perrier,Dominique,Medium
10223,37,100.00,1,3965.66,2004-02-20 00:00:00,Shipped,1,2,2004,Motorcycles,95,S10_1678,"Australian Collectors, Co.",03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,APAC,Ferguson,Peter,Medium
10275,45,92.83,1,4177.35,2004-07-23 00:00:00,Shipped,3,7,2004,Motorcycles,95,S10_1678,La Rochelle Gifts,40.67.8555,"67, rue des Cinquante Otages",None,Nantes,None,44000,France,EMEA,Labrune,Janine,Medium
10299,23,100.00,9,2597.39,2004-09-30 00:00:00,Shipped,3,9,2004,Motorcycles,95,S10_1678,"Toys of Finland, Co.",90-224 8555,Keskuskatu 45,None,Helsinki,None,21240,Finland,EMEA,Karttunen,Matti,Small
10309,41,100.00,5,4394.38,2004-10-15 00:00:00,Shipped,4,10,2004,Motorcycles,95,S10_1678,Baane Mini Imports,07-98 9555,Erling Skakkes gate 78,None,Stavern,None,4110,Norway,EMEA,Bergulfsen,Jonas,Medium
10341,41,100.00,9,7737.93,2004-11-24 00:00:00,Shipped,4,11,2004,Motorcycles,95,S10_1678,Salzburg Collectables,6562-9555,Geislweg 14,None,Salzburg,None,5020,Austria,EMEA,Pipps,Georg,Large


In [6]:
%%sql

SELECT 'customers' AS table_name, COUNT(*) FROM customers
UNION ALL SELECT 'products', COUNT(*) FROM products
UNION ALL SELECT 'orders', COUNT(*) FROM orders
UNION ALL SELECT 'order_items', COUNT(*) FROM order_items;


table_name,count
customers,92
products,109
orders,307
order_items,2823


**TOP REVENUE PRODUCTS**   
​Which PRODUCTLINE generates the highest total SALES? Order the results from highest to lowest

In [ ]:
%%sql
SELECT p.product_line, SUM(oi.sales) total_revenue
FROM  order_items oi
INNER JOIN products p
ON oi.product_code = p.product_code 
GROUP BY  p.product_line
ORDER BY total_revenue DESC
;

product_line,total_revenue
Classic Cars,3919615.66
Vintage Cars,1903150.84
Motorcycles,1166388.34
Trucks and Buses,1127789.84
Planes,975003.57
Ships,714437.13
Trains,226243.47


**Unique COUNTRY values and their the total number of orders placed by each country.**

In [9]:
%%sql
SELECT  c.country,  
        COUNT(DISTINCT o.order_number) total_order_placed
FROM customers c
INNER JOIN orders o
ON c.customer_id = o.customer_id
GROUP BY c.country
ORDER BY total_order_placed DESC
;

country,total_order_placed
USA,112
France,37
Spain,36
Australia,19
UK,13
Singapore,9
Norway,9
Finland,9
Italy,8
Austria,7


**Deal Size Distribution: What is the average QUANTITYORDERED and average SALES value for each DEALSIZE (Small, Medium, Large)?**

In [10]:
%%sql
SELECT o.deal_size,
       ROUND(AVG(oi.quantity_ordered), 2) AS avg_qty_ordered,
       ROUND(AVG(oi.sales), 2) AS avg_sales
FROM order_items oi
INNER JOIN orders o 
ON oi.order_number = o.order_number
GROUP BY o.deal_size;

deal_size,avg_qty_ordered,avg_sales
Small,34.41,3379.68
Large,39.19,4615.50
Medium,35.45,3641.64


**Find all details of customers whose total lifetime spending (SALES) is strictly greater than the global average lifetime spending across all customers.**

In [11]:
%%sql
WITH customer_lifetime_spending AS (
  SELECT o.customer_id,
         SUM(oi.sales) AS total_spending
  FROM order_items oi
  INNER JOIN orders o 
  ON oi.order_number = o.order_number
  GROUP BY o.customer_id 
)
SELECT  c.customer_name, 
        c.city,
        c.country,
        ROUND(cls.total_spending, 2) AS lifetime_value
FROM customers c 
INNER JOIN customer_lifetime_spending cls ON c.customer_id = cls.customer_id
WHERE cls.total_spending > (SELECT AVG(total_spending)
                            FROM customer_lifetime_spending)
ORDER BY lifetime_value DESC;

customer_name,city,country,lifetime_value
Euro Shopping Channel,Madrid,Spain,912294.11
Mini Gifts Distributors Ltd.,San Rafael,USA,654858.06
"Australian Collectors, Co.",Melbourne,Australia,200995.41
Muscle Machine Inc,NYC,USA,197736.94
La Rochelle Gifts,Nantes,France,180124.90
"Dragon Souveniers, Ltd.",Singapore,Singapore,172989.68
Land of Toys Inc.,NYC,USA,164069.44
The Sharp Gifts Warehouse,San Jose,USA,160010.27
"AV Stores, Co.",Manchester,UK,157807.81
"Anna's Decorations, Ltd",North Sydney,Australia,153996.13


**Find all orders where the actual selling price  was lower than the suggested retail price (MSRP). Include a column showing the calculated loss per unit.**

In [13]:
%%sql
SELECT oi.order_number,
      oi.price_each,
      p.product_line,
      p.msrp,
      ROUND((p.msrp - oi.price_each), 2) AS loss_per_unit
FROM  order_items oi
INNER JOIN products p 
ON oi.product_code = p.product_code
WHERE oi.price_each < p.msrp
ORDER BY loss_per_unit DESC
LIMIT 10;

order_number,price_each,product_line,msrp,loss_per_unit
10348,52.36,Classic Cars,207,154.64
10336,57.22,Classic Cars,207,149.78
10325,64.00,Classic Cars,207,143.00
10395,69.12,Classic Cars,207,137.88
10329,71.47,Classic Cars,194,122.53
10380,47.18,Vintage Cars,168,120.82
10375,76.00,Classic Cars,194,118.00
10339,76.67,Motorcycles,193,116.33
10332,52.67,Vintage Cars,168,115.33
10341,79.65,Classic Cars,194,114.35


**​Peak Sales DAY: find which specific DAY  generated the highest total revenue.**

In [34]:
%%sql
SELECT  EXTRACT(YEAR FROM o.order_date) AS order_year,
        EXTRACT(MONTH FROM o.order_date) AS order_month,
        EXTRACT(DAY FROM o.order_date) AS order_day,
        SUM(oi.sales) AS total_revenue
FROM orders o 
INNER JOIN order_items oi
ON o.order_number = oi.order_number
GROUP BY EXTRACT(YEAR FROM o.order_date),
        EXTRACT(MONTH FROM o.order_date),
        EXTRACT(DAY FROM o.order_date) 
ORDER BY total_revenue DESC
LIMIT 1;


order_year,order_month,order_day,total_revenue
2004,11,24,137644.72


**​Peak Sales MONTH: find which specific MONTH generated the highest total revenue.**

In [30]:
%%sql
SELECT EXTRACT(YEAR FROM o.order_date) AS order_year,
       EXTRACT(MONTH FROM o.order_date) AS order_month,
       SUM(oi.sales) AS total_revenue
FROM orders o
INNER JOIN order_items oi ON o.order_number = oi.order_number
GROUP BY EXTRACT(YEAR FROM o.order_date), EXTRACT(MONTH FROM o.order_date)
ORDER BY total_revenue DESC
LIMIT 1;

order_year,order_month,total_revenue
2004,11,1089048.01


**​Year-Over-Year Moving Ranks: Within each YEAR_ID, rank the product lines (PRODUCTLINE) based on their total sales revenue using DENSE_RANK()**

In [ ]:
%%sql
-- #TOTAL SALES PER CATEGORY PER YEAR

WITH annual_sales AS (
  SELECT o.year_id,
         p.product_line,
         SUM(oi.sales) AS total_revenue
FROM order_items oi
INNER JOIN orders o ON oi.order_number = o.order_number
INNER JOIN products p ON oi.product_code = p.product_code
GROUP BY o.year_id, p.product_line
),

-- #RANK FOR EACH CATEGORY WITHIN ITS RESPECTIVE YEAR

yearly_ranks AS (
  SELECT year_id,
          Product_line,
          total_revenue,
          RANK() OVER (
            PARTITION BY year_id
            ORDER BY total_revenue DESC
          ) AS current_rank
  FROM annual_sales
)

--  #FETCH PRIOR YEAR RANK AND COMPUTE YOY MOVE

SELECT  year_id,
        product_line,
        ROUND(total_revenue, 2) AS total_revenue,
        current_rank,
        LAG(current_rank) OVER (
          PARTITION BY product_line
          ORDER BY year_id
        ) AS prior_year_rank,

-- #CALCULATE POSITION CHANGE(prior rank - current rank)
-- #positive number = gained positions(climbed up)
-- #negative number - lost positions

(LAG(current_rank) OVER (
  PARTITION BY product_line
  ORDER BY year_id
) - current_rank) AS rank_change
FROM yearly_ranks
ORDER BY product_line, year_id;

year_id,product_line,total_revenue,current_rank,prior_year_rank,rank_change
2003,Classic Cars,1484785.29,1,None,None
2004,Classic Cars,1762257.09,1,1,0
2005,Classic Cars,672573.28,1,1,0
2003,Motorcycles,370895.58,4,None,None
2004,Motorcycles,560545.23,3,4,1
2005,Motorcycles,234947.53,3,3,0
2003,Planes,272257.60,5,None,None
2004,Planes,502671.80,5,5,0
2005,Planes,200074.17,4,5,1
2003,Ships,244821.09,6,None,None


**Monthly Cumulative Running Total: Create a query that displays the chronological progression of sales by month. For each row, include a running total column that calculates the cumulative sales up to that month.**

In [19]:
%%sql
-- #Calculate total sales for each month
WITH monthly_sales AS (
    SELECT 
        TO_CHAR(o.order_date, 'YYYY-MM') AS year_month,
        SUM(oi.sales) AS month_revenue
    FROM orders o
    INNER JOIN order_items oi ON o.order_number = oi.order_number
    GROUP BY TO_CHAR(o.order_date, 'YYYY-MM')
)

-- #Apply running total function
SELECT 
    year_month,
    ROUND(month_revenue, 2) AS monthly_revenue,
    ROUND(
        SUM(month_revenue) OVER (ORDER BY year_month ASC), 
        2
    ) AS running_total_revenue
FROM monthly_sales
ORDER BY year_month ASC
LIMIT 10;

year_month,monthly_revenue,running_total_revenue
2003-01,129753.60,129753.60
2003-02,140836.19,270589.79
2003-03,174504.90,445094.69
2003-04,201609.55,646704.24
2003-05,192673.11,839377.35
2003-06,168082.56,1007459.91
2003-07,187731.88,1195191.79
2003-08,197809.30,1393001.09
2003-09,263973.36,1656974.45
2003-10,568290.97,2225265.42


**Customer Re-engagement : Using LAG() or LEAD() find the date of a customer’s current order vs. the date of their previous order. Calculate the number of days that passed between their purchases to see how frequently top customers return.**

In [ ]:
%%sql
-- #Look back at the customer's previous order date
WITH customer_order_history AS (
    SELECT 
        customer_id,
        order_number,
        order_date,
        LAG(order_date) OVER (
            PARTITION BY customer_id 
            ORDER BY order_date ASC
        ) AS previous_order_date
    FROM orders
)

-- #Calculate the day gap between purchases
SELECT 
    customer_id,
    order_number,
    order_date,
    previous_order_date,
    (order_date - previous_order_date) AS days_since_last_order
FROM customer_order_history
WHERE previous_order_date IS NOT NULL    -- Excluding their first purchase
ORDER BY customer_id, order_date
LIMIT 10;

customer_id,order_number,order_date,previous_order_date,days_since_last_order
1,10305,2004-10-13 00:00:00,2004-08-27 00:00:00,"47 days, 0:00:00"
2,10265,2004-07-02 00:00:00,2003-11-21 00:00:00,"224 days, 0:00:00"
2,10415,2005-05-09 00:00:00,2004-07-02 00:00:00,"311 days, 0:00:00"
3,10252,2004-05-26 00:00:00,2004-01-15 00:00:00,"132 days, 0:00:00"
3,10402,2005-04-07 00:00:00,2004-05-26 00:00:00,"316 days, 0:00:00"
4,10319,2004-11-03 00:00:00,2004-04-20 00:00:00,"197 days, 0:00:00"
5,10259,2004-06-15 00:00:00,2004-02-04 00:00:00,"132 days, 0:00:00"
5,10288,2004-09-01 00:00:00,2004-06-15 00:00:00,"78 days, 0:00:00"
5,10409,2005-04-23 00:00:00,2004-09-01 00:00:00,"234 days, 0:00:00"
6,10170,2003-11-04 00:00:00,2003-10-21 00:00:00,"14 days, 0:00:00"


**Create a Common Table Expression (CTE) that flags all orders with a STATUS of 'Disputed', 'On Hold', or 'Cancelled'. Use that CTE in a main query to calculate the total amount of "at-risk" or "lost" revenue per COUNTRY.**

In [ ]:
%%sql
-- #CTE to isolate at-risk or lost orders
WITH risky_orders AS (
    SELECT 
        order_number,
        customer_id,
        status
    FROM orders
    WHERE status IN ('Disputed', 'On Hold', 'Cancelled')
)

-- #Calculate lost/at-risk revenue per country
SELECT 
    c.country,
    ro.status,
    ROUND(SUM(oi.sales), 2) AS at_risk_revenue
FROM risky_orders ro
INNER JOIN order_items oi ON ro.order_number = oi.order_number
INNER JOIN customers c ON ro.customer_id = c.customer_id
GROUP BY c.country, ro.status
ORDER BY at_risk_revenue DESC;

country,status,at_risk_revenue
USA,On Hold,152718.98
UK,Cancelled,50408.25
Spain,Cancelled,50010.65
Sweden,Cancelled,48710.92
USA,Cancelled,45357.66
Spain,Disputed,31821.90
Sweden,On Hold,26260.21
Denmark,Disputed,26012.87
Australia,Disputed,14378.09


**Rank each order's revenue within its own product line**

In [ ]:
%%sql 
SELECT
    product_line,
    order_number,
    order_revenue,
    RANK() OVER (PARTITION BY product_line ORDER BY order_revenue DESC) AS rank_in_line,
    ROW_NUMBER() OVER (PARTITION BY product_line ORDER BY order_revenue DESC) AS row_num_in_line
FROM (
      SELECT
          p.product_line,
           oi.order_number,
           SUM(oi.sales) AS order_revenue
      FROM order_items oi
      JOIN products p ON oi.product_code = p.product_code
      GROUP BY p.product_line, oi.order_number
     )  revenue_by_order
ORDER BY product_line, rank_in_line
LIMIT 10;

product_line,order_number,order_revenue,rank_in_line,row_num_in_line
Classic Cars,10287,67281.01,1,1
Classic Cars,10212,65165.17,2,2
Classic Cars,10310,64573.72,3,3
Classic Cars,10192,63981.45,4,4
Classic Cars,10181,60795.84,5,5
Classic Cars,10419,59475.10,6,6
Classic Cars,10266,56421.65,7,7
Classic Cars,10225,50432.55,8,8
Classic Cars,10253,50408.25,9,9
Classic Cars,10204,49798.07,10,10


**Revenue trends over time (monthly)**

In [23]:
%%sql
SELECT
    DATE_TRUNC('month', o.order_date) AS month,
    ROUND(SUM(oi.sales), 2) AS monthly_revenue
FROM orders o
JOIN order_items oi ON o.order_number = oi.order_number
GROUP BY month
ORDER BY month;

month,monthly_revenue
2003-01-01 00:00:00,129753.60
2003-02-01 00:00:00,140836.19
2003-03-01 00:00:00,174504.90
2003-04-01 00:00:00,201609.55
2003-05-01 00:00:00,192673.11
2003-06-01 00:00:00,168082.56
2003-07-01 00:00:00,187731.88
2003-08-01 00:00:00,197809.30
2003-09-01 00:00:00,263973.36
2003-10-01 00:00:00,568290.97


**Customer purchasing behavior: average order value and deal size mix**

In [24]:
%%sql
SELECT
    c.customer_name,
    COUNT(DISTINCT o.order_number) AS number_of_orders,
    ROUND(AVG(oi.sales), 2) AS avg_line_value,
    ROUND(SUM(oi.sales), 2) AS lifetime_value,
    MODE() WITHIN GROUP (ORDER BY o.deal_size) AS most_common_deal_size
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_number = oi.order_number
GROUP BY c.customer_name
ORDER BY lifetime_value DESC
LIMIT 10;

customer_name,number_of_orders,avg_line_value,lifetime_value,most_common_deal_size
Euro Shopping Channel,26,3522.37,912294.11,Medium
Mini Gifts Distributors Ltd.,17,3638.10,654858.06,Small
"Australian Collectors, Co.",5,3654.46,200995.41,Small
Muscle Machine Inc,4,4119.52,197736.94,Medium
La Rochelle Gifts,4,3398.58,180124.90,Medium
"Dragon Souveniers, Ltd.",5,4023.02,172989.68,Medium
Land of Toys Inc.,4,3348.36,164069.44,Small
The Sharp Gifts Warehouse,4,4000.26,160010.27,Small
"AV Stores, Co.",3,3094.27,157807.81,Small
"Anna's Decorations, Ltd",4,3347.74,153996.13,Medium


**Which product line generates the most revenue per territory (top product line per territory)**

In [25]:
%%sql
SELECT territory, product_line, line_revenue
FROM (
    SELECT
        c.territory,
        p.product_line,
        SUM(oi.sales) AS line_revenue,
        RANK() OVER (PARTITION BY c.territory ORDER BY SUM(oi.sales) DESC) AS rnk
    FROM order_items oi
    JOIN orders o ON oi.order_number = o.order_number
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN products p ON oi.product_code = p.product_code
    WHERE c.territory IS NOT NULL
    GROUP BY c.territory, p.product_line
) ranked
WHERE rnk = 1
ORDER BY line_revenue DESC;

territory,product_line,line_revenue
EMEA,Classic Cars,2086994.66
AMER,Classic Cars,1406261.44
APAC,Classic Cars,244758.07
Japan,Classic Cars,181601.49
